# CAMP — run Convergent Adaptive Monge Patching on your STL meshes

This notebook takes STL outputs (e.g. from **vascular-cut**) and runs them through **CAMP**, producing a per-face patch partition and the integrated–Gaussian–curvature shape metric for each mesh.

**Pipeline (fully automatic):**

> `STL`  →  weld duplicated corners  →  **uniform ACVD remesh**  →  **CAMP**  →  outputs

The uniform remesh is the *only* preprocessing step. Everything else is CAMP with the parameters you choose below.

**How to use:** edit the one **Settings** cell (①), then run every cell top to bottom (Kernel ▸ Restart & Run All). Results land in `OUT_DIR` — one subfolder per mesh, plus a `summary.csv`. Each mesh also gets an interactive viewer with an **adjustable integrated-K color axis**.

> First time? Install dependencies with `pip install -r requirements.txt` (or `conda env create -f environment.yml`).

In [ ]:
# --- setup: locate and import the camp_toolkit package -------------------------
import sys
from pathlib import Path

# find the folder that contains camp_toolkit/ (works wherever Jupyter was launched)
_here = Path.cwd()
for _cand in [_here, *_here.parents]:
    if (_cand / "camp_toolkit" / "__init__.py").exists():
        sys.path.insert(0, str(_cand)); break

import camp_toolkit
from camp_toolkit import run_camp, view, DEFAULT_PARAMS

print("camp_toolkit loaded from:", Path(camp_toolkit.__file__).parent)
print("default CAMP parameters :", DEFAULT_PARAMS)

## ① Settings — the only cell you need to edit

In [ ]:
# ============================== EDIT ME ==============================

# 1) INPUT — a single .stl file, OR a folder containing .stl files
INPUT = r"path\to\your\stls_or_one.stl"

# 2) OUTPUT — where results are written (one subfolder per mesh + summary.csv)
OUT_DIR = r"camp_outputs"

# 3) REMESH DENSITY — target face count for the uniform remesh before CAMP
#      ~50_000  : plain thoracic / abdominal aortas (no branches)
#      ~100_000 : aortas WITH branches  (recommended for vascular-cut outputs)
#    Use the SAME value for every mesh in a study so results are comparable.
TARGET_FACES = 100_000

# 4) TRIM — symmetric quantile trim on integrated-K for the k_fluct metric.
#    0.01 = production.  Affects only the metric, not the saved patches or plots.
TRIM_FRAC = 0.01

# 5) CAMP QUALITY PARAMETERS — defaults are the project operating point.
#    Usually leave these alone (see the reference at the bottom for what they do).
CAMP_PARAMS = dict(
    R2_thresh      = 0.9,    # min variance a patch's quadratic fit must explain
    eta_thresh     = 0.075,  # max curvature variation allowed within a patch
    min_patch_size = 18,     # smallest patch (vertices); never split below this
    seed_patches   = 100,    # number of initial KMeans seeds
)

# ====================================================================
print("INPUT        :", INPUT)
print("OUT_DIR      :", OUT_DIR)
print("TARGET_FACES :", TARGET_FACES)
print("TRIM_FRAC    :", TRIM_FRAC)
print("CAMP_PARAMS  :", CAMP_PARAMS)

## ② Run CAMP

Welds → remeshes → partitions every mesh. Progress prints as it goes; expect roughly 1–3 minutes per mesh at 100k faces.

In [ ]:
summary = run_camp(
    inputs       = INPUT,
    out_dir      = OUT_DIR,
    target_faces = TARGET_FACES,
    trim_frac    = TRIM_FRAC,
    params       = CAMP_PARAMS,
    save_html    = True,   # interactive 3-D viewer per mesh (adjustable color axis)
    save_png     = True,   # static image per mesh (needs a GL context; skipped if unavailable)
)
summary

## ③ Browse the results — integrated-K color axis

The first successful mesh is shown inline below, colored by **integrated Gaussian curvature**. The view has built-in controls (no server needed):

- **color min / color max sliders** (bottom) — drag to set the color axis so the map reads well
- **colorscale dropdown** (top-right) — Plasma, Viridis, Turbo, …
- **integrated K / patches** buttons (top-left) — toggle the curvature map vs. random per-patch colors

Drag to rotate. Every mesh also has a standalone `*_patches.html` (same controls) you can open in a browser and share.

In [ ]:
ok = summary[summary["status"] == "ok"]
if len(ok):
    stem = ok.iloc[0]["stem"]
    view(Path(OUT_DIR) / stem).show()          # integrated K, adjustable color axis
else:
    print("No successful meshes — check the 'error' column in `summary`.")

In [ ]:
# Color by a different per-patch metric (auto color axis).  Options:
#   "integrated_K" (default) · "avg_kg" (K at patch center) · "eta" (curvature
#   variation) · "R2" (fit quality)
# To FIX the axis for comparing meshes on the same scale, pass cmin=..., cmax=...
#   e.g.  view(Path(OUT_DIR)/stem, cmin=-5, cmax=40).show()
if len(ok):
    view(Path(OUT_DIR) / stem, metric="eta", colorscale="Viridis").show()

In [ ]:
# every file produced this run
for p in sorted(Path(OUT_DIR).rglob("*")):
    if p.is_file():
        print(p.relative_to(OUT_DIR))

---
## Reference

### What you get, per mesh (`OUT_DIR/<mesh_name>/`)

| file | contents |
|---|---|
| `<name>_facepatches.csv` | `face_id, patch_id` — the CAMP patch label for every face of the remeshed mesh |
| `<name>_patch_metrics.csv` | one row per patch: `avg_kg, area_mm2, integrated_K, eta, R2, kept` |
| `<name>_camp.npz` | vertices `V`, faces `F`, `face_labels`, and all per-patch arrays (for re-plotting) |
| `<name>_patches.html` | interactive 3-D viewer, **adjustable integrated-K color axis** |
| `<name>_integratedK.png` | static curvature image |

and `OUT_DIR/summary.csv` — one row per mesh: `n_patches`, `k_fluct`, `surface_area_mm2`, `euler_in`/`euler_remesh`, `pct_at_min_size`, `runtime_s`, …

**`k_fluct`** is the headline shape descriptor: the area-weighted variance of integrated Gaussian curvature across patches. Higher = more heterogeneous surface shape.

### Visualization & color axis

Every `*_patches.html` opens in a browser with color-axis sliders, a colorscale dropdown, and an integrated-K ↔ patches toggle — no server or install needed. Re-open or re-color any result from Python:

```python
from camp_toolkit import view
view("camp_outputs/<name>").show()                     # integrated K, adjustable axis
view("camp_outputs/<name>", cmin=-5, cmax=40).show()   # fixed axis (compare meshes)
view("camp_outputs/<name>", metric="eta").show()       # or avg_kg / R2
```

### The knobs

- **`TARGET_FACES`** — remesh density. More faces let CAMP subdivide more finely in tight regions (ostia, arch) at the cost of runtime. **Keep it fixed across a cohort** so `k_fluct` is comparable. Guidance: ~50k plain aortas, ~100k branched.
- **`TRIM_FRAC`** — drops the most extreme integrated-K patches (both tails) before computing `k_fluct`, so a few sliver patches at cut edges don't dominate. Only affects the metric.
- **`R2_thresh` (0.9)** — a patch is “good” only if its quadratic fit explains ≥90% of its height variance. Lower → coarser patches.
- **`eta_thresh` (0.075)** — max curvature variation within a patch (≈ patch radius ≤ 0.39× local radius of curvature). Lower → finer patches.
- **`min_patch_size` (18)** — vertex floor; patches are never split below this (curvature estimates get too noisy on tiny patches).
- **`seed_patches` (100)** — initial KMeans seeds; the final patch count is data-driven, so this rarely matters.